# backward-fn-signature — ex2: write negative_back and exp_back — back-fn signature, two ops

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backward-fn-signature`. Running the final beacon cell reports progress against the `Backprop: backward fn signature` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: backward fn signature` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-fn-signature`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-fn-signature"
DD_SUBTOPIC = "Backprop: backward fn signature"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Backward-fn signature — quick refresher

In a manual autograd, every forward op `f(x, y, ...) -> out` is paired with **one backward fn per input position**. The canonical signature is:

```python
def f_back<i>(grad_out, out, *args, **kwargs):
    """dL/dargs[i] given dL/dout, cached out, and the original args."""
    ...
```

- `grad_out` — the upstream gradient `dL/dout`, same shape as `out`.
- `out` — the cached forward output (so you don't recompute).
- `*args, **kwargs` — the original forward inputs (one of them is the input you're differentiating w.r.t.).

The fn returns `dL/dargs[i]`, **same shape as `args[i]`**. For elementwise ops the local Jacobian is diagonal so you just multiply `grad_out` by the elementwise derivative — no actual matrix is materialized.

### Exercise 2 — write negative_back and exp_back — back-fn signature, two ops

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the uniform (grad_out, out, x) backward-fn signature across two ops — one that ignores `out`, one that requires it — to internalize the calling convention.
> Keywords: backward-fn, negative, exp, uniform-signature
> ```

**KCs targeted:** `backward-fn-signature`, `back-fn-uses-cached-out`

Implement TWO backward fns that share the same signature:

**1. `negative_back(grad_out, out, x)`** — gradient of `out = -x`.
   - Math: `d(-x)/dx = -1`, so `dL/dx = -grad_out`.
   - Notice you don't need `x` OR `out` — pure sign flip of grad_out.

**2. `exp_back(grad_out, out, x)`** — gradient of `out = exp(x)`.
   - Math: `d(exp x)/dx = exp(x) = out`, so `dL/dx = grad_out * out`.
   - This is the contrasting case — `out` IS used (and faster than      re-computing `exp(x)`).

**The point of this drill.** Both fns take `(grad_out, out, x)`. One ignores most of its inputs, the other uses `out` directly. The uniform calling convention is what makes the BACK_FUNCS dispatcher work — no signature gymnastics.

Return tensors with the SAME shape as `x` for both fns.

In [ ]:
def negative_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d(-x)/dx = -1, so dL/dx = -grad_out. `out` and `x` unused.
    return -grad_out


def exp_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d(exp x)/dx = exp(x) = out (cached). dL/dx = grad_out * out.
    return grad_out * out


<details><summary>Solution</summary>

```python
def negative_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d(-x)/dx = -1, so dL/dx = -grad_out. `out` and `x` unused.
    return -grad_out


def exp_back(grad_out: Tensor, out: Tensor, x: Tensor) -> Tensor:
    # d(exp x)/dx = exp(x) = out (cached). dL/dx = grad_out * out.
    return grad_out * out
```

**The two halves of why `out` is in the signature.** `negative_back` ignores `out`. `exp_back` *requires* `out` (and benefits from it being cached — no second `exp` call). Most back fns fall into one of these two buckets. Some, like `sigmoid_back`, even use `out` instead of `x` because the formula is cleaner: `d/dx sigmoid(x) = sigmoid(x)*(1 - sigmoid(x)) = out * (1 - out)`.

**Why a uniform signature beats per-op signatures.** ARENA's reverse pass is one line: `back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)`. If each back fn had a bespoke signature, the dispatcher would need switch logic per op — at which point you've rebuilt torch.autograd's C++ dispatcher in slow Python.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()